### For our Project (Working! -> Just paste all of the rest things in Terminal)

First, we will use PDBFixer to clean the structure and add missing hydrogens at a physiological pH (7.4): 

### TO DO
1. In 'congig.txt', find center(x,y,z), select 'box size'
2. Prepare Receptor
   - Download the receptor (MMP-1), change the name in the second cell
   - Use PDBFixer to clean the structure. For us, we have to add Ca2+, Zn2+
   - In the Third cell, change 'A:203' to the correct part according to where Zn2+ is in!
   - Download 20 ligands we will use in the 'ligands_raw' folder as ligand_raw_{#}

In [4]:
import csv
import pandas as pd

def extract_result(input_file, csv_file):
    with open(input_file, 'r') as f:
        lines = f.readlines()

    data_rows = []
    start_collecting = False
    
    for line in lines:
        if '-----+' in line:
            start_collecting = True
            continue
        
        if start_collecting and line.strip():
            parts = line.split()
            if parts and parts[0].isdigit():
                data_rows.append(parts[1:])

    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['affinity', 'rmsd_lb', 'rmsd_ub'])
        writer.writerows(data_rows)

def best_affinity(table_file):
    df = pd.read_csv(table_file)
    first_affinity = df['affinity'].iloc[0]
    return first_affinity

In [5]:
table_list = []

for i in range(1, 21):
    extract_result(f'./results/log_ligand_{i}.txt', f'table_ligand_{i}.csv')
    #table_list.append(f'table_ligand_{i}.csv')

In [6]:
best_affinity = {}

for i in range(1, 21):
    df = pd.read_csv(f'table_ligand_{i}.csv')
    best_affinity.update({f'ligand_{i}' : df['affinity'].iloc[0]})

print(best_affinity)

sort_affinity = dict(sorted(best_affinity.items(), key=lambda item: item[1]))

print(sort_affinity)

IndexError: single positional indexer is out-of-bounds

# will sort rmsd_ub..

### From lab 5

In [ ]:
mk_prepare_receptor.py -i beta2ar_receptorH.pdb -o flex_receptor -p -v \ 
#mk_prepare_receptor.py -i beta2ar_receptorH.pdb (name of the receptor we are using) -o flex_receptor -p -v \ 
--box_center 64.610 17.949 12.734 \ #Find the point of Zn2+
--box_size 25.0 25.0 25.0 \ #Set the size based on the paper
-f A:203 -a #Find 1~3 places

In [ ]:
vina --config config.txt \
--receptor flex_receptor_rigid.pdbqt \
--flex flex_receptor_flex.pdbqt \
--exhaustiveness 18 \
--out output_flex.pdbqt | tee log_flex.txt

In [ ]:
mk_export.py output_flex.pdbqt -s output_flex_poses_withflex.sdf -k

In [ ]:
vina --config config.txt --out output.pdbqt | tee log.txt

In [ ]:
import sys
print(sys.executable)

### Trying to do with Python..

In [ ]:
import os

ligand_dir = './ligands'
result_dir = './results'
config_file = 'config.txt'

if not os.path.exists(result_dir):
    os.makedirs(result_dir)

In [ ]:
!mk_prepare_receptor.py -i beta2ar_receptorH.pdb -o flex_receptor -p -v --box_center 64.610 17.949 12.734 --box_size 25.0 25.0 25.0 -f A:203 -a 

Checkpoint: you have both flex_receptor_rigid.pdbqt and flex_receptor_flex.pdbqt, plus
flex_receptor.box.txt.

mk_prepare_receptor.py -i beta2ar_receptorH.pdb -o flex_receptor -p -v \ 
#mk_prepare_receptor.py -i beta2ar_receptorH.pdb (name of the receptor we are using) -o flex_receptor -p -v \ 
--box_center 64.610 17.949 12.734 \ #Find the point of Zn2+
--box_size 25.0 25.0 25.0 \ #Set the size based on the paper
-f A:203 -a #Find 1~3 places

In [ ]:
for i in range(1, 2):

    !vina --config config.txt \
          --receptor flex_receptor_rigid.pdbqt \
          --flex flex_receptor_flex.pdbqt \
          --ligand ./ligands/ligand_{i}.pdbqt \
          --exhaustiveness 18 \
          --out ./results/output_ligand_{i}.pdbqt | tee ./results/log_ligand_{i}.txt

    !mk_export.py ./results/output_ligand_{i}.pdbqt -s ./results/output_visual_ligand_{i}.sdf -k

In [ ]:
for i in range(1, 2):
    ligand_name = f"ligand_{i}.pdbqt"
    input_path = os.path.join(ligand_dir, ligand_name)

    output_pdbqt = os.path.join(result_dir, f"output_ligand_{i}.pdbqt")
    log_file = os.path.join(result_dir, f"log_ligand_{i}.txt")
    output_sdf = os.path.join(result_dir, f"output_visual_ligand_{i}.sdf")

    !vina --config {config_file} \
          --receptor flex_receptor_rigid.pdbqt \
          --flex flex_receptor_flex.pdbqt \
          --ligand {input_path} \
          --exhaustiveness 18 \
          --out {output_pdbqt} | tee {log_file}

    !mk_export.py {output_pdbqt} -s {output_sdf} -k

In [ ]:
for i in range(1, 21):
    ligand_name = f"ligand_{i}.pdbqt"
    input_path = os.path.join(ligand_dir, ligand_name)

    output_pdbqt = os.path.join(result_dir, f"output_ligand_{i}.pdbqt")
    log_file = os.path.join(result_dir, f"log_ligand_{i}.txt")
    output_sdf = os.path.join(result_dir, f"output_visual_ligand_{i}.sdf")

    !vina --config {config_file} \
          --receptor flex_receptor_rigid.pdbqt \
          --flex flex_receptor_flex.pdbqt \
          --ligand {input_path} \
          --exhaustiveness 18 \
          --out {output_pdbqt} | tee {log_file}

    !mk_export.py {output_pdbqt} -s {output_sdf} -k

Checkpoint: output_ligand{i}.pdbqt and log_ligand{i}.txt, output_visual_ligand{i} were created.

In [7]:
## Research and Background
Protein structure: The 3D protein structure used for this study is (PDB ID: 4AUO) and was obtained from the RCSB Protein Bank. This structure shows the structure of MMP-1  in a helical collagen peptide. It also depicts the active enzymes and how inhibitors interact with it 
20 Types of MMP-1 Inhibitors that interact with Zn^2+
Batimastat (BB-94)
Marimastat (BB-2516)
Ilomastat (GM6001)
Prinomastat (AG3340)
CGS 27023A
Ro 32-3555
RS-130830
TAPI-1  
Rebimastat (BMS-275291)  
Tanomastat (BAY 12-9566)  
MMI-270  
PD-166793 
Doxycycline
Minocycline  
Ac-Pro-Leu-Gly-hydroxamate
Collagen-mimicking peptide inhibitors
Phosphinic peptide inhibitors  
NNGH
SB-3CT
Trocade (cipemastat)  
### The ligand bind differently to Zn^2 and represent different kinds like Hydroxamate- based compounds, carboxylate inhibitors, tetracyclines, and peptide-based inhibitors

### References
Whittaker et al., Chemical Reviews
Moss et al., Nature Reviews Drug Discovery
JNCI: Development of Matrix metalloproteinase Inhibitors in Cancer Therapy

                                                                                                   